# Clean Full50 S-JEPA PreLocal Downstream Augmentation
Matched control versus training-only montage rotation and Gaussian noise using the authors' pinned released S-JEPA weights. Original validation and test trials are never augmented.

# 1. Setup

In [ ]:
import builtins
import hashlib
import json
import os
import platform
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import torch

WORKING_DIR = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(WORKING_DIR / 'src' / 'liu2024'))
import liu2024_prelocal_clean as clean
import liu2024_prelocal_augmentation as aug

mne.set_log_level('WARNING')
print(f'Python: {sys.version.split()[0]} | platform: {platform.platform()}')
print(f'Working directory: {WORKING_DIR}')

# 2. Configuration
## 2.1 Clean Full50 Defaults
The direct-run defaults are a validation-only one-subject smoke. The batch configuration freezes the Full50 comparison.

In [ ]:
CONFIG = {
    # Paths / run identity
    'artifact_dir': str(WORKING_DIR / 'artifacts' / 'liu2024-sjepa-prelocal-clean-gacl-full50'),
    'source_extract_dir': str(WORKING_DIR / 'liu2024_data' / 'liu2024_figshare' / 'sourcedata'),
    'experiment_name': 'sjepa_prelocal_clean_gacl_validation_smoke',
    'config_note': 'Safe default: checkpoint, transform, and one-subject data validation only.',

    # Dataset and preprocessing
    'subjects_to_use': [1],
    'exclude_subjects': [],
    'allow_subset_for_smoke': True,
    'validation_only': True,
    'target_sfreq': 128,
    'mi_window_seconds': 4.0,
    'average_reference': True,
    'bandpass_hz': [0.5, 40.0],
    'normalization_mode': 'none',
    'normalization_eps': 1e-6,

    # Model and fixed conditions
    'model_name': 'SignalJEPA_PreLocal',
    'pretrained_repo_id': 'braindecode/signal-jepa_without-chans',
    'pretrained_revision': '213876ea30f0764fd25c055efcb55d1d1652a371',
    'strategy': 'new',
    'conditions': ['control', 'gacl_rotation_noise'],
    'augmentation': {
        'rotation_axis': 'z', 'rotation_max_degrees': 10.0,
        'rotation_probability': 0.5, 'rotation_spherical_splines': True,
        'noise_std_microvolts': 2.0, 'noise_probability': 1.0,
    },
    'fixed_materialized_descendants': 0,

    # Evaluation and training
    'cv_folds': 5, 'cv_seed': 2026, 'val_fraction': 0.2, 'val_seed': 2026,
    'batch_size': 16, 'num_workers': 0, 'n_epochs': 2,
    'early_stopping_patience': 50, 'early_stopping_threshold': 0.0,
    'learning_rate': 0.0005, 'weight_decay': 0.0,

    # Reproducibility and diagnostics
    'seed': 2026, 'set_seed': True, 'collapse_threshold': 0.875,
    'bootstrap_iterations': 1000,
}


In [ ]:
aug.validate_config(CONFIG, require_full50=not CONFIG.get('allow_subset_for_smoke', False))
if int(CONFIG.get('fixed_materialized_descendants', -1)) != 0:
    raise RuntimeError('Fixed augmented descendants are prohibited; use fresh training-only views.')
print(json.dumps(CONFIG, indent=2))

## 2.2 Artifact Creation and Logging Init

In [ ]:
def create_run_id():
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    config_hash = hashlib.md5(json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:8]
    return f'{timestamp}_{config_hash}'

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG['artifact_dir']) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)
LOG_PATH = ARTIFACT_DIR / 'run.log'
_LOG_FILE_HANDLE = open(LOG_PATH, 'a', buffering=1, encoding='utf-8', errors='replace')

def _safe_write_text(stream, text):
    try: stream.write(text)
    except UnicodeEncodeError:
        encoding = getattr(stream, 'encoding', None) or 'utf-8'
        stream.write(text.encode(encoding, errors='replace').decode(encoding, errors='replace'))

def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop('sep', ' '); end = kwargs.pop('end', '\n'); flush = kwargs.pop('flush', False); file = kwargs.pop('file', None)
    message = sep.join(str(arg) for arg in args); target = sys.stdout if file is None else file
    stamped = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}" if message else ''
    _safe_write_text(target, stamped + end); _safe_write_text(_LOG_FILE_HANDLE, stamped + end)
    if flush: target.flush(); _LOG_FILE_HANDLE.flush()

builtins.print = _timestamped_print
config_path = ARTIFACT_DIR / 'config.json'
config_path.write_text(json.dumps(CONFIG, indent=2) + '\n')
print(f'Run ID:     {RUN_ID}')
print(f'Artifacts:  {ARTIFACT_DIR}')
print(f'Config:     {config_path}')

## 2.3 Reproducibility

In [ ]:
def resolve_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built(): return torch.device('mps')
    if torch.cuda.is_available(): return torch.device('cuda')
    return torch.device('cpu')
DEVICE = resolve_device(); BASE_SEED = int(CONFIG['seed'])
if CONFIG['set_seed']: clean.seed_everything(BASE_SEED)
print(f'Using device: {DEVICE} | seed: {BASE_SEED}')

# 3. Load and Prepare Data
## 3.1 Exact-Marker Independent-Trial Preprocessing

In [ ]:
source_root = Path(CONFIG['source_extract_dir'])
paths = sorted(source_root.glob('sub-*/sub-*_task-motor-imagery_eeg.mat'))
requested = None if CONFIG['subjects_to_use'] is None else {int(x) for x in CONFIG['subjects_to_use']}
excluded = {int(x) for x in CONFIG['exclude_subjects']}
paths = [p for p in paths if (requested is None or clean.subject_id_from_path(p) in requested) and clean.subject_id_from_path(p) not in excluded]
if not paths: raise FileNotFoundError(f'No selected MAT files under {source_root}')
SUBJECT_DATA = {}; inventory_rows = []; marker_rows = []
for path in paths:
    subject = clean.load_subject(path); sid = str(subject['subject_id'])
    x, records = clean.preprocess_subject(subject, CONFIG)
    SUBJECT_DATA[sid] = {'x': x, 'y': subject['labels'].copy()}
    inventory_rows.append({'subject_id': sid, 'source_path': subject['path'], 'source_sha256': subject['source_sha256'], 'preprocessed_shape': list(x.shape), 'class_counts': np.bincount(subject['labels'], minlength=2).tolist()})
    marker_rows.extend(records)
SUBJECTS = sorted(SUBJECT_DATA, key=int)
if not CONFIG.get('allow_subset_for_smoke', False) and SUBJECTS != [str(i) for i in range(1, 51)]: raise AssertionError('Full50 subject contract failed')
subject_inventory_path = ARTIFACT_DIR / 'subject_inventory.csv'; marker_inventory_path = ARTIFACT_DIR / 'trial_marker_inventory.csv'
pd.DataFrame(inventory_rows).to_csv(subject_inventory_path, index=False); pd.DataFrame(marker_rows).to_csv(marker_inventory_path, index=False)
print(f'Loaded {len(SUBJECTS)} subjects and {len(marker_rows)} exact-marker trials')

# 4. Model
## 4.1 Pinned Released PreLocal Encoder and Transform Preflight

In [ ]:
probe_model, MODEL_AUDIT = clean.build_model(CONFIG, 512)
expected_trainable = {'spatial_conv.1.weight', 'spatial_conv.1.bias', 'final_layer.1.weight', 'final_layer.1.bias'}
if set(MODEL_AUDIT['trainable']) != expected_trainable: raise AssertionError(MODEL_AUDIT['trainable'])
probe_initial_hash = clean.state_hash(probe_model); del probe_model
probe_x = SUBJECT_DATA[SUBJECTS[0]]['x'][:4]; probe_y = SUBJECT_DATA[SUBJECTS[0]]['y'][:4]
transforms_a, _ = aug.build_fold_transforms(CONFIG, BASE_SEED); transforms_b, _ = aug.build_fold_transforms(CONFIG, BASE_SEED)
batch_a = next(iter(aug.make_train_loader(probe_x, probe_y, 4, BASE_SEED, transforms_a)))
batch_b = next(iter(aug.make_train_loader(probe_x, probe_y, 4, BASE_SEED, transforms_b)))
assert batch_a[0].shape == torch.Size([4, 29, 512]) and torch.isfinite(batch_a[0]).all()
assert torch.equal(batch_a[0], batch_b[0]) and torch.equal(batch_a[1], batch_b[1])
assert not torch.equal(batch_a[0], torch.from_numpy(probe_x))
print(f'Preflight passed: trainable={MODEL_AUDIT["trainable"]} initial_hash={probe_initial_hash[:12]}')

# 5. Training
## 5.1 Matched Control and Training-Only Augmentation

In [ ]:
FOLD_RESULTS = {condition: [] for condition in CONFIG['conditions']}
fold_dir = ARTIFACT_DIR / 'fold_shards'; fold_dir.mkdir(exist_ok=True)
if not CONFIG.get('validation_only', False):
    for sid in SUBJECTS:
        data = SUBJECT_DATA[sid]
        for split in clean.make_outer_splits(data['y'], CONFIG):
            pair = {}
            for condition in CONFIG['conditions']:
                print(f'Subject {sid} fold {split["fold_id"]}/{CONFIG["cv_folds"]} condition={condition}')
                result = aug.run_fold(condition, int(sid), data['x'], data['y'], split, CONFIG, DEVICE)
                pair[condition] = result; FOLD_RESULTS[condition].append(result)
                shard = fold_dir / f'{condition}_sub-{int(sid):02d}_fold-{int(split["fold_id"]):02d}.json'
                temp = shard.with_suffix('.tmp'); temp.write_text(json.dumps(result, indent=2) + '\n'); os.replace(temp, shard)
                print(f"  BA={result['balanced_accuracy']:.3f} best_epoch={result['best_epoch']} pred={result['prediction_histogram']}")
            aug.assert_matched_pair(pair['control'], pair['gacl_rotation_noise'])
else:
    print('Validation-only mode: no folds trained and no accuracy result produced.')

# 6. Results
## 6.1 Completion, Exact-Once Aggregation, and Paired Statistics

In [ ]:
labels_by_subject = {sid: SUBJECT_DATA[sid]['y'] for sid in SUBJECTS}
CONDITION_OUTPUTS = {}; PAIRED_ROWS = []; PAIRED_STATS = {}; GLOBAL_METRICS = {'validation_only': bool(CONFIG.get('validation_only', False))}
if not CONFIG.get('validation_only', False):
    expected_folds = len(SUBJECTS) * int(CONFIG['cv_folds'])
    for condition in CONFIG['conditions']:
        rows = FOLD_RESULTS[condition]
        if len(rows) != expected_folds: raise AssertionError(f'{condition}: incomplete folds')
        subjects, metrics, predictions = clean.aggregate(rows, labels_by_subject)
        metrics['single_class_folds'] = sum(r['collapse_diagnostics']['single_class_prediction'] for r in rows)
        metrics['near_collapse_folds'] = sum(r['collapse_diagnostics']['near_collapse_7_of_8'] for r in rows)
        metrics['predicted_class_1_fraction'] = float(np.mean([p['y_pred'] for p in predictions]))
        CONDITION_OUTPUTS[condition] = {'subjects': subjects, 'metrics': metrics, 'predictions': predictions}
    if CONDITION_OUTPUTS['control']['metrics']['split_hash'] != CONDITION_OUTPUTS['gacl_rotation_noise']['metrics']['split_hash']: raise AssertionError('Condition split hashes differ')
    PAIRED_ROWS, PAIRED_STATS = aug.paired_statistics(CONDITION_OUTPUTS['control']['subjects'], CONDITION_OUTPUTS['gacl_rotation_noise']['subjects'], CONFIG['bootstrap_iterations'], BASE_SEED)
    GLOBAL_METRICS = {'methods': {c: CONDITION_OUTPUTS[c]['metrics'] for c in CONFIG['conditions']}, 'paired_augmented_minus_control': PAIRED_STATS, 'n_subjects': len(SUBJECTS), 'n_predictions_per_condition': len(CONDITION_OUTPUTS['control']['predictions']), 'exact_once_oof': True}
print(json.dumps(GLOBAL_METRICS, indent=2))

## 6.2 Save Artifacts and Visualizations

In [ ]:
artifact_paths = {'config': str(config_path), 'run_log': str(LOG_PATH), 'subject_inventory': str(subject_inventory_path), 'marker_inventory': str(marker_inventory_path)}
global_metrics_path = ARTIFACT_DIR / 'global_metrics.json'; global_metrics_path.write_text(json.dumps(GLOBAL_METRICS, indent=2) + '\n'); artifact_paths['global_metrics'] = str(global_metrics_path)
if not CONFIG.get('validation_only', False):
    for condition in CONFIG['conditions']:
        output = CONDITION_OUTPUTS[condition]
        cv_path = ARTIFACT_DIR / f'cv_results_{condition}.json'; subject_path = ARTIFACT_DIR / f'subject_metrics_{condition}.json'; prediction_path = ARTIFACT_DIR / f'trial_predictions_{condition}.csv'
        cv_path.write_text(json.dumps(FOLD_RESULTS[condition], indent=2) + '\n'); subject_path.write_text(json.dumps(output['subjects'], indent=2) + '\n')
        pd.DataFrame(output['predictions']).sort_values(['subject_id', 'trial_index']).to_csv(prediction_path, index=False)
        artifact_paths[f'cv_results_{condition}'] = str(cv_path); artifact_paths[f'subject_metrics_{condition}'] = str(subject_path); artifact_paths[f'trial_predictions_{condition}'] = str(prediction_path)
    paired_path = ARTIFACT_DIR / 'paired_subject_results.csv'; paired_stats_path = ARTIFACT_DIR / 'paired_statistics.json'
    pd.DataFrame(PAIRED_ROWS).sort_values('subject_id', key=lambda s: s.astype(int)).to_csv(paired_path, index=False); paired_stats_path.write_text(json.dumps(PAIRED_STATS, indent=2) + '\n')
    artifact_paths['paired_subject_results'] = str(paired_path); artifact_paths['paired_statistics'] = str(paired_stats_path)
    plot_path = ARTIFACT_DIR / 'paired_subject_balanced_accuracy.png'; frame = pd.DataFrame(PAIRED_ROWS).sort_values('subject_id', key=lambda s: s.astype(int)); positions = np.arange(len(frame)); fig, ax = plt.subplots(figsize=(14, 5)); ax.plot(positions, 100 * frame['control_balanced_accuracy'], 'o-', label='Control'); ax.plot(positions, 100 * frame['augmented_balanced_accuracy'], 'o-', label='Rotation + noise'); ax.axhline(50, color='black', ls='--', lw=1); ax.set(xticks=positions, xticklabels=frame['subject_id'], xlabel='Subject', ylabel='Exactly-once OOF BA (%)', ylim=(0, 100)); ax.legend(); fig.tight_layout(); fig.savefig(plot_path, dpi=160); plt.close(fig); artifact_paths['paired_plot'] = str(plot_path)
module_paths = [Path(clean.__file__).resolve(), Path(aug.__file__).resolve()]
run_metadata_path = ARTIFACT_DIR / 'run_metadata.json'; completion_path = ARTIFACT_DIR / 'COMPLETED.json'
artifact_paths['fold_shards'] = str(fold_dir); artifact_paths['run_metadata'] = str(run_metadata_path); artifact_paths['completion'] = str(completion_path)
run_metadata = {'run_id': RUN_ID, 'artifact_dir': str(ARTIFACT_DIR), 'experiment_name': CONFIG['experiment_name'], 'config_note': CONFIG['config_note'], 'subjects': SUBJECTS, 'model_name': CONFIG['model_name'], 'pretrained_repo_id': CONFIG['pretrained_repo_id'], 'pretrained_revision': CONFIG['pretrained_revision'], 'conditions': CONFIG['conditions'], 'augmentation': CONFIG['augmentation'], 'fixed_materialized_descendants': 0, 'preprocessing_contract': 'independent complete-trial average reference/FIR 0.5-40 Hz/resample, exact marker-2 [0,4)s, microvolts', 'implementation_files': {str(p): clean.sha256_file(p) for p in module_paths}, 'model_load_audit': MODEL_AUDIT, 'global_metrics': GLOBAL_METRICS, 'leakage_assertions': {'augmentation_inner_training_only': True, 'validation_augmented': False, 'test_augmented': False, 'outer_test_used_for_fit': False, 'outer_test_used_for_selection': False, 'exact_once_oof': not CONFIG.get('validation_only', False)}, 'artifacts': artifact_paths}
run_metadata_path.write_text(json.dumps(run_metadata, indent=2) + '\n')
completion = {'completed': True, 'validation_only': bool(CONFIG.get('validation_only', False)), 'run_id': RUN_ID, 'n_subjects': len(SUBJECTS), 'n_folds_per_condition': len(FOLD_RESULTS['control']), 'n_predictions_per_condition': 0 if CONFIG.get('validation_only', False) else len(CONDITION_OUTPUTS['control']['predictions'])}
completion_path.write_text(json.dumps(completion, indent=2) + '\n')
print(f'Run metadata saved to: {run_metadata_path}')
print(f'\nAll artifacts in: {ARTIFACT_DIR}')
try: _LOG_FILE_HANDLE.close()
except Exception: pass